# Differentiable HBV–precipitation-fusion demonstration

This example uses **fully synthetic, non-geographic data** (4 mock subbasins, 2 fictitious gauges). It demonstrates the model API, not a reproduction of the Jialing River study. Start Jupyter in the repository root and execute the cells in order.

In [ ]:
from pathlib import Path
import torch
from src.data import load_model_data
from src.fusion import Rain_HBV, Static_HBV
from src.losses import station_loss, calculate_process_loss

data = load_model_data("examples/synthetic_data/tiny_demo.npz")
print("Basins:", len(data.topo_ids), "Days:", len(data.dates), "Stations:", data.station_names)

In [ ]:
model = Rain_HBV(24, 4, 9, data.downstream_index, data.routing_order)
qsim, state, params, river_uh, weights, pfused, aet, comp = model(
    data.x_hbv, data.x_rain, data.x_river, data.forcing, data.area, return_comp=True)
print("Q shape:", tuple(qsim.shape), "Weights shape:", tuple(weights.shape))
print("Fusion sums to one:", torch.allclose(weights.sum(-1), torch.ones_like(pfused), atol=1e-5))
print("River UH sums to one:", torch.allclose(river_uh.sum(-1), torch.ones(len(data.topo_ids)), atol=1e-5))

In [ ]:
mask = torch.ones(len(data.dates), dtype=torch.bool)
q_gauges = qsim.index_select(0, data.station_indices)
flow_loss, _, _, nse, kge = station_loss(q_gauges, data.qobs_station, 
    torch.isfinite(data.qobs_station), mask)
process_loss, swe_r, sm_r, et_r = calculate_process_loss(
    comp, aet, data.swe_obs, data.sm_obs, data.e_obs, mask)
loss = 0.8 * flow_loss + 0.2 * process_loss  # DEMO only: no RCI geometry
loss.backward()
print(f"Demo loss={loss.item():.3f}; flow NSE={nse.item():.3f}; finite gradient=", 
      torch.isfinite(model.rain_net[-1].weight.grad).all().item())

**Notes:** The synthetic streamflow and process observations are fabricated and are not designed to agree with a calibrated model. Low hydrological scores are expected. To train the submitted-paper configuration, use your authorized private data and `configs/gauged.yaml` / `configs/pub.yaml`. The paper notebooks' process and RCI losses are documented in `docs/experiments.md`.